# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202405_Flood_Brasil'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 8 .tif files in the S3 bucket.


['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 8
  - Total size: 0.43 GB

📁 Cached files (first 10):
  - drcs_activations/202402_Fire_Guatemala/sentinel2/swir/S2B_shortwaveInfrared_20240223_162159_T15PYS.tif (86.3 MB)
  - drcs_activations/202402_Fire_Guatemala/sentinel2/true/S2B_trueColor_20240223_162159_T15PYS.tif (345.1 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T020715Z_20240206T130219Z_S1A_30_v0.1_B01_WTR.tif (1.6 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T140849Z_20240206T130818Z_S1A_30_v0.1_B01_WTR.tif (0.9 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T020715Z_20240206T022545Z_S1A_30_v0.1_B01_WTR.tif (1.6 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T140849Z_20240206T134347Z_S1A_30_v0.1_B01_WTR.tif (0.9 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t035_20240206T0

(8, 459943829)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [13]:
def create_cog_filename_floodmap(f, EVENT_NAME):
    """Create COG filename for FloodMap files, moving FloodMap before dates and ensuring earlier date first."""
    f2 = Path(f).stem
    
    # Check if it has FloodMap at the end
    if '_FloodMap' in f2:
        # Remove _FloodMap from the end
        f2_no_floodmap = f2.replace('_FloodMap', '')
        
        # Split by underscore
        parts = f2_no_floodmap.split('_')
        
        # Find the date range part (YYYYMMDD-YYYYMMDD)
        date_index = None
        for i, part in enumerate(parts):
            if '-' in part and len(part) == 17 and part[:8].isdigit() and part[9:].isdigit():
                date_index = i
                date_range = part
                # Extract dates
                date1, date2 = date_range.split('-')
                
                # Compare dates and swap if needed (ensure earlier date first)
                if date1 > date2:
                    date1, date2 = date2, date1
                
                # Format dates
                formatted_dates = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}day_{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
                break
        
        if date_index is not None:
            # Reconstruct with FloodMap before dates
            base_parts = parts[:date_index]
            new_name = '_'.join(base_parts) + '_FloodMap_' + formatted_dates
            cog_filename = f'{EVENT_NAME}_{new_name}.tif'
        else:
            # Fallback if format is unexpected
            cog_filename = f'{EVENT_NAME}_{f2}.tif'
    else:
        # Fallback if no FloodMap found
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

filter_str = 'FloodMap'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_floodmap(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-04-21day_2024-05-06.tif


In [14]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_floodmap, 
                                target_dir = "HLS/aria", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-04-21day_2024-05-06.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif
   Output filename: 202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-04-21day_2024-05-06.tif
   [MEMORY] Initial: 289.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [NODATA] Source nodata value: 0
   [CHUNKS] Processing 104 chunks (13x8)
   [BAND 1/1] Processing...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=6684/1000000
            Estimated data coverage: 2.2% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpgwnmhofr_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp2yxw4ivd.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-04-21day_2024-05-06.tif
   [MEMORY] Final: 654.6 MB (Change: +365.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_DSWx_HLS_FloodMap_2024-04-21day_2024-05-06.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T17:05:41.535961


In [15]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [18]:
# Define filename creator functions for different file types

def create_cog_filename_dswx_hls_mosaic(f, EVENT_NAME):
    """Create COG filename for DSWx-HLS mosaic files, moving suffix before date."""
    f2 = Path(f).stem
    
    # Split by hyphen to separate the main parts
    parts = f2.split('-')
    
    # Find the date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Get everything before the date
        prefix_parts = parts[:date_index]
        # Get everything after the date
        suffix_parts = parts[date_index + 1:]
        
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Reconstruct: prefix + suffix + date
        prefix = '-'.join(prefix_parts)
        suffix = '-'.join(suffix_parts) if suffix_parts else ""
        
        if suffix:
            # Replace hyphens with underscores for consistency
            prefix = prefix.replace('-', '_')
            suffix = suffix.replace('-', '_')
            cog_filename = f'{EVENT_NAME}_{prefix}_{suffix}_{formatted_date}_day.tif'
        else:
            prefix = prefix.replace('-', '_')
            cog_filename = f'{EVENT_NAME}_{prefix}_{formatted_date}_day.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}_day.tif'
    
    return cog_filename


filter_str = 'S2B_L8'  # This will catch files starting with dates in the DSWx folder

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_mosaic(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06_day.tif


In [19]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_dswx_hls_mosaic, 
                                target_dir = "HLS/opera", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/opera

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06_day.tif
   [MEMORY] Initial: 655.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
   [CHUNKS] Processing 440 chunks (22x20)
   [BAND 1/1] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999996/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpp4742qty_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpal8o2wn9.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06_day.tif
   [MEMORY] Final: 907.6 MB (Change: +252.4 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2B_L8_mosaic_2024-05-06_day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/opera/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/opera/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T17:07:46.126412


In [24]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [20]:
# Define filename creator functions for different file types
import re 
def create_cog_filename_dswx_hls_timestamp(f, EVENT_NAME):
    """Create COG filename for DSWx-HLS files with timestamp, moving date to end."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find the part with timestamp (YYYYMMDDTHHMMSSZ format)
    timestamp_index = None
    timestamp_str = None
    
    for i, part in enumerate(parts):
        if 'T' in part and 'Z' in part and len(part) >= 16:
            # Extract just the date part from timestamp
            timestamp_index = i
            timestamp_str = part
            break
    
    if timestamp_index is not None and timestamp_str:
        # Extract date from timestamp (YYYYMMDD from YYYYMMDDTHHMMSSZ)
        date_part = timestamp_str[:8]
        time_part = timestamp_str[9:15]  # HHMMSS
        
        # Format date and time
        formatted_datetime = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}T{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}Z"
        
        # Remove timestamp from parts
        parts.pop(timestamp_index)
        
        # Reconstruct filename with date at the end
        base_name = '_'.join(parts).replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{base_name}_{formatted_datetime}.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}day.tif'
    
    return cog_filename

pattern = re.compile(r'(?=.*S2A)(?=.*mosaic)')
filter_ = [f for f in keys if re.search(pattern, f)]

# Test functions
print("Testing WM filename:")

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_timestamp(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif


In [23]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_dswx_hls_timestamp, 
                                target_dir = "HLS/opera", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-18T13:22:31Z.tif
  202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/opera

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/3] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif
   Output filename: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_mosaic_2024-04-21T13:31:51Z.tif
   [MEMORY] Initial: 970.2 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Opt

Band 1:  40%|████      | 169/420 [00:06<00:08, 28.06chunks/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f7849bbaae0>>
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
                                                             

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=253, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmphwhdx1n4_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=252, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpp_0zhnm4_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp3yhddd5m.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif
   [MEMORY] Final: 930.6 MB (Change: -64.4 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_L3_DSWx_HLS_S2A_30_mosaic_2024-05-21T13:31:51Z.tif

✅ Batch processing complete: 3 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/opera/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/opera/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T17:11:47.889189


In [22]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [25]:

def create_cog_filename_dswx_hls_archive(f, EVENT_NAME):
    """Create COG filename for archive DSWx-HLS files, moving date to the end."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find the date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Remove date from parts
        parts.pop(date_index)
        
        # Reconstruct with date at the end
        base_name = '_'.join(parts).replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{base_name}_{formatted_date}_day.tif'
    else:
        # Fallback if format is unexpected
        f2_cleaned = f2.replace('-', '_')
        cog_filename = f'{EVENT_NAME}_{f2_cleaned}_day.tif'
    
    return cog_filename


filter_str = 'archive'
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_dswx_hls_archive(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21_day.tif
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06_day.tif


In [26]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_dswx_hls_archive, 
                                target_dir = "HLS/archive", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21_day.tif
  202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/archive

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/2] Processing: drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif
   Output filename: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-04-21_day.tif
   [MEMORY] Initial: 930.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
   [CHUNKS] Processing 252 chunks (18x14)
   [

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999996/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpn_y275ot_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpjnho4x5h.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=253, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmper4n1l5a_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpzt9kakvp.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/archive/202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06_day.tif
   [MEMORY] Final: 927.6 MB (Change: +0.2 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_OPERA_DSWx_HLS_S2B_Mosaic_NoSnowIce_2024-05-06_day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/archive/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/archive/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T17:12:35.544248


In [27]:
keys

['drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif',
 'drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif']

In [28]:
def create_cog_filename_flood_depth(f, EVENT_NAME):
    """Create COG filename for flood depth files, adding event date from EVENT_NAME."""
    f2 = Path(f).stem
    
    # Extract date from EVENT_NAME (format: YYYYMM_EventType_Location)
    event_parts = EVENT_NAME.split('_')
    if event_parts and len(event_parts[0]) == 6 and event_parts[0].isdigit():
        event_date = event_parts[0]  # YYYYMM
        # Format as YYYY-MM
        formatted_date = f"{event_date[:4]}{event_date[4:6]}"
        
        # Create filename with event date at the end
        cog_filename = f'{EVENT_NAME}_{f2}_{formatted_date}month.tif'
    else:
        # Fallback if EVENT_NAME doesn't have expected format
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename


filter_str = 'flood_depth'
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_flood_depth(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif


In [29]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_flood_depth, 
                                target_dir = "GoogleEarth", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/GoogleEarth

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/1] Processing: drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_GEE_FwDET_Brazil.tif
   Output filename: 202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif
   [MEMORY] Initial: 927.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=3.3999999521443642e+38, center sample non-zero=217344/1000000
            Estimated data coverage: 6.8% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Us

Reading input: /tmp/tmpgs_kmwsb_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp5_ur_eda.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/GoogleEarth/202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif
   [MEMORY] Final: 946.1 MB (Change: +18.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_FwDET_GEE_FwDET_Brazil_202405month.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/GoogleEarth/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/GoogleEarth/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T17:12:50.695298


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [30]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 946.1 MB
  Available memory: 26932.4 MB
  Memory percent used: 14.8%
